In [7]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pathlib import Path
 
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit only this section
# ══════════════════════════════════════════════════════════════════════════════
 
DATA_DIR   = Path("./results")          # directory containing CSV files
OUTPUT_DIR = Path("plots")      # output directory (created automatically)
FMT        = "png"              # "png" or "pdf"
 
FILES = {
    "lr":      ("custom_cnn_lr_history.csv",              "custom_cnn_lr_results.csv"),
    "bs":      ("custom_cnn_bs_history.csv",              "custom_cnn_bs_results.csv"),
    "dropout": ("custom_cnn_phase3_history.csv",          "custom_cnn_phase3_results.csv"),
    "wd":      ("custom_cnn_phase4_history.csv",          "custom_cnn_phase4_results.csv"),
    "aug_std": ("custom_cnn_phase5_history.csv",          "custom_cnn_phase5_results.csv"),
    "aug_adv": ("custom_cnn_phase6_history.csv",          "custom_cnn_phase6_results.csv"),
    "aug_mix": ("custom_cnn_phase7_history.csv",          "custom_cnn_phase7_results.csv"),
    "fewshot": ("custom_cnn_phase8_fewshot_history.csv",  "custom_cnn_phase8_fewshot_results.csv"),
}
 
# ══════════════════════════════════════════════════════════════════════════════
# STYLE
# ══════════════════════════════════════════════════════════════════════════════
 
PALETTE = ["#2166ac", "#d6604d", "#4dac26", "#8073ac", "#e08214", "#1a9850"]
 
STYLE = {
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": ":",
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
}
 
# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
 
def load_history(phase: str) -> pd.DataFrame:
    path = DATA_DIR / FILES[phase][0]
    return pd.read_csv(path)
 
def load_results(phase: str) -> pd.DataFrame:
    path = DATA_DIR / FILES[phase][1]
    return pd.read_csv(path)
 
def epoch_stats(df: pd.DataFrame, config: str) -> pd.DataFrame:
    """Compute per-epoch mean and std across seeds for a given config."""
    sub = df[df["config"] == config]
    return (
        sub.groupby("epoch")
           .agg(
               train_loss_mean=("train_loss", "mean"), train_loss_std=("train_loss", "std"),
               val_loss_mean  =("val_loss",   "mean"), val_loss_std  =("val_loss",   "std"),
               train_f1_mean  =("train_f1",   "mean"), train_f1_std  =("train_f1",   "std"),
               val_f1_mean    =("val_f1",      "mean"), val_f1_std    =("val_f1",     "std"),
           )
           .reset_index()
    )
 
def short_label(config: str, keep_suffix: bool = False) -> str:
    """Shorten a config string to its most informative parameter segment."""
    parts = config.split("_")
    meaningful = [p for p in parts if any(k in p for k in
                  ["LR=", "BS=", "dropout=", "wd=", "aug="])]
    if keep_suffix:
        return "_".join(meaningful)
    return meaningful[-1] if meaningful else config
 
def save(fig: plt.Figure, name: str) -> None:
    OUTPUT_DIR.mkdir(exist_ok=True)
    path = OUTPUT_DIR / f"{name}.{FMT}"
    fig.savefig(path)
    print(f"  v {path}")
    plt.close(fig)
 
def _learning_curves_ax(ax, df, configs, metric, show_train, labels=None):
    """Draw mean +/- std learning curves onto a given Axes object."""
    y_col = f"val_{metric}"
    y_t   = f"train_{metric}"
    for i, cfg in enumerate(configs):
        s   = epoch_stats(df, cfg)
        c   = PALETTE[i % len(PALETTE)]
        lbl = (labels[i] if labels else short_label(cfg))
 
        ax.plot(s["epoch"], s[f"{y_col}_mean"], color=c, lw=2, label=f"Val - {lbl}")
        ax.fill_between(s["epoch"],
                        s[f"{y_col}_mean"] - s[f"{y_col}_std"],
                        s[f"{y_col}_mean"] + s[f"{y_col}_std"],
                        color=c, alpha=0.15)
        if show_train:
            ax.plot(s["epoch"], s[f"{y_t}_mean"],
                    color=c, lw=1.3, ls="--", label=f"Train - {lbl}")
            ax.fill_between(s["epoch"],
                            s[f"{y_t}_mean"] - s[f"{y_t}_std"],
                            s[f"{y_t}_mean"] + s[f"{y_t}_std"],
                            color=c, alpha=0.07)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
 
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 & 2 - LR sweep + BS sweep
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_hyperparams_sweep():
    """2x2 panel: F1 and Loss learning curves for the LR sweep and BS sweep."""
    df_lr = load_history("lr")
    df_bs = load_history("bs")
    cfgs_lr = sorted(df_lr["config"].unique())
    cfgs_bs = sorted(df_bs["config"].unique())
 
    labels_lr = [re.search(r"LR=([^_]+)", c).group(1) for c in cfgs_lr]
    labels_bs = [re.search(r"BS=([^_]+)", c).group(1) for c in cfgs_bs]
 
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        fig.suptitle("Phase 1-2: Effect of Learning Rate and Batch Size - Custom CNN",
                     fontsize=14, y=1.01)
 
        ax = axes[0, 0]
        _learning_curves_ax(ax, df_lr, cfgs_lr, "f1", show_train=False, labels=labels_lr)
        ax.set_title("LR sweep - Val Macro F1")
        ax.set_ylabel("Macro F1")
        ax.legend(title="LR")
 
        ax = axes[0, 1]
        _learning_curves_ax(ax, df_lr, cfgs_lr, "loss", show_train=True, labels=labels_lr)
        ax.set_title("LR sweep - Train / Val Loss")
        ax.set_ylabel("Loss")
        ax.legend(title="LR")
 
        ax = axes[1, 0]
        _learning_curves_ax(ax, df_bs, cfgs_bs, "f1", show_train=False, labels=labels_bs)
        ax.set_title("BS sweep - Val Macro F1")
        ax.set_ylabel("Macro F1")
        ax.set_xlabel("Epoch")
        ax.legend(title="BS")
 
        ax = axes[1, 1]
        _learning_curves_ax(ax, df_bs, cfgs_bs, "loss", show_train=True, labels=labels_bs)
        ax.set_title("BS sweep - Train / Val Loss")
        ax.set_ylabel("Loss")
        ax.set_xlabel("Epoch")
        ax.legend(title="BS")
 
        fig.tight_layout()
        save(fig, "01_lr_bs_sweep")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3 & 4 - Dropout + Weight Decay
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_regularization():
    """2x2 panel: F1 curves and train-val gap for Dropout and Weight Decay sweeps."""
    df_do = load_history("dropout")
    df_wd = load_history("wd")
    cfgs_do = sorted(df_do["config"].unique())
    cfgs_wd = sorted(df_wd["config"].unique())
 
    labels_do = [re.search(r"dropout=([^_]+)", c).group(1) for c in cfgs_do]
    labels_wd = [re.search(r"wd=([^_]+)",      c).group(1) for c in cfgs_wd]
 
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        fig.suptitle("Phase 3-4: Regularization - Dropout and Weight Decay",
                     fontsize=14, y=1.01)
 
        ax = axes[0, 0]
        _learning_curves_ax(ax, df_do, cfgs_do, "f1", show_train=False, labels=labels_do)
        ax.set_title("Dropout sweep - Val Macro F1")
        ax.set_ylabel("Macro F1")
        ax.legend(title="Dropout")
 
        ax = axes[0, 1]
        ax.axhline(0, color="gray", lw=1, alpha=0.5)
        for i, (cfg, lbl) in enumerate(zip(cfgs_do, labels_do)):
            s   = epoch_stats(df_do, cfg)
            gap = s["train_f1_mean"] - s["val_f1_mean"]
            ax.plot(s["epoch"], gap, color=PALETTE[i], lw=2, label=lbl)
            ax.fill_between(s["epoch"], 0, gap, color=PALETTE[i], alpha=0.10)
        ax.set_title("Dropout - Train-Val gap (F1)")
        ax.set_ylabel("Train F1 - Val F1")
        ax.legend(title="Dropout")
 
        ax = axes[1, 0]
        _learning_curves_ax(ax, df_wd, cfgs_wd, "f1", show_train=False, labels=labels_wd)
        ax.set_title("Weight Decay sweep - Val Macro F1")
        ax.set_ylabel("Macro F1")
        ax.set_xlabel("Epoch")
        ax.legend(title="WD")
 
        ax = axes[1, 1]
        ax.axhline(0, color="gray", lw=1, alpha=0.5)
        for i, (cfg, lbl) in enumerate(zip(cfgs_wd, labels_wd)):
            s   = epoch_stats(df_wd, cfg)
            gap = s["train_f1_mean"] - s["val_f1_mean"]
            ax.plot(s["epoch"], gap, color=PALETTE[i], lw=2, label=lbl)
            ax.fill_between(s["epoch"], 0, gap, color=PALETTE[i], alpha=0.10)
        ax.set_title("Weight Decay - Train-Val gap (F1)")
        ax.set_ylabel("Train F1 - Val F1")
        ax.set_xlabel("Epoch")
        ax.legend(title="WD")
 
        fig.tight_layout()
        save(fig, "02_regularization")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 5-7 - Augmentation techniques
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_augmentations():
    """
    Top row: F1 curves for standard augmentations, CutMix, CutOut/Random.
    Bottom: horizontal bar chart comparing all augmentation strategies by Test F1.
    """
    df5 = load_history("aug_std")
    df6 = load_history("aug_adv")
    df7 = load_history("aug_mix")
 
    r5 = load_results("aug_std")
    r6 = load_results("aug_adv")
    r7 = load_results("aug_mix")
 
    cfgs5 = sorted(df5["config"].unique())
    cfgs6 = sorted(df6["config"].unique())
    cfgs7 = sorted(df7["config"].unique())
 
    labels5 = [re.search(r"aug=(.+)$", c).group(1) for c in cfgs5]
    labels6 = [re.search(r"aug=(.+)$", c).group(1) for c in cfgs6]
    labels7 = [re.search(r"aug=(.+)$", c).group(1) for c in cfgs7]
 
    with plt.rc_context(STYLE):
        fig = plt.figure(figsize=(14, 11))
        gs  = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.32)
        fig.suptitle("Phase 5-7: Data Augmentation Techniques - Custom CNN", fontsize=14)
 
        ax1 = fig.add_subplot(gs[0, 0])
        _learning_curves_ax(ax1, df5, cfgs5, "f1", show_train=False, labels=labels5)
        ax1.set_title("Standard augmentations")
        ax1.set_ylabel("Val Macro F1")
        ax1.legend(fontsize=9)
 
        ax2 = fig.add_subplot(gs[0, 1])
        _learning_curves_ax(ax2, df6, cfgs6, "f1", show_train=False, labels=labels6)
        ax2.set_title("CutMix (alpha=1 vs alpha=4)")
        ax2.legend(fontsize=9)
 
        ax3 = fig.add_subplot(gs[0, 2])
        _learning_curves_ax(ax3, df7, cfgs7, "f1", show_train=False, labels=labels7)
        ax3.set_title("CutOut and Random augmentation")
        ax3.legend(fontsize=9)
 
        ax4 = fig.add_subplot(gs[1, :])
        all_results = pd.concat([r5, r6, r7], ignore_index=True)
 
        def aug_label(cfg):
            m = re.search(r"aug=(.+)$", cfg)
            return m.group(1) if m else cfg
 
        all_results["aug"] = all_results["config"].apply(aug_label)
        all_results = all_results.sort_values("test_f1_mean")
 
        colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(all_results))]
        bars = ax4.barh(all_results["aug"], all_results["test_f1_mean"],
                        xerr=all_results["test_f1_std"], capsize=4,
                        color=colors_bar, alpha=0.82,
                        error_kw=dict(lw=1.3, capthick=1.3))
 
        for bar, (_, row) in zip(bars, all_results.iterrows()):
            ax4.text(row["test_f1_mean"] + row["test_f1_std"] + 0.002,
                     bar.get_y() + bar.get_height() / 2,
                     f'{row["test_f1_mean"]:.4f} +/-{row["test_f1_std"]:.4f}',
                     va="center", fontsize=9)
 
        ax4.set_xlabel("Test Macro F1")
        ax4.set_title("All augmentation strategies - Test F1 (mean +/- std, n=5 seeds)")
        ax4.set_xlim(all_results["test_f1_mean"].min() - 0.05,
                     all_results["test_f1_mean"].max() + 0.06)
 
        save(fig, "03_augmentations")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# HEATMAP - hyperparameter phases 1-4
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_heatmap_hyperparam():
    """
    One column per hyperparameter phase; best configuration highlighted with a box.
    Cell values: Test F1 mean +/- std.
    """
    phases = {
        "LR sweep":     load_results("lr"),
        "BS sweep":     load_results("bs"),
        "Dropout":      load_results("dropout"),
        "Weight Decay": load_results("wd"),
    }
 
    def param_label(cfg, phase_name):
        patterns = {
            "LR sweep":     r"LR=([^_]+)",
            "BS sweep":     r"BS=([^_]+)",
            "Dropout":      r"dropout=([^_]+)",
            "Weight Decay": r"wd=([^_]+)",
        }
        m = re.search(patterns[phase_name], cfg)
        return m.group(1) if m else cfg
 
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, len(phases), figsize=(12, 4.5),
                                 gridspec_kw={"wspace": 0.4})
        fig.suptitle("Heatmap - Test Macro F1 (mean +/- std, n=5 seeds)", fontsize=13)
 
        for ax, (phase_name, df) in zip(axes, phases.items()):
            df = df.copy()
            df["label"] = df["config"].apply(lambda c: param_label(c, phase_name))
            df = df.sort_values("test_f1_mean", ascending=True)
 
            vals = df["test_f1_mean"].values.reshape(-1, 1)
            stds = df["test_f1_std"].values
            lbls = df["label"].tolist()
 
            vmin, vmax = vals.min() - 0.005, vals.max() + 0.005
            ax.imshow(vals, cmap="RdYlGn", aspect="auto", vmin=vmin, vmax=vmax)
 
            ax.set_xticks([])
            ax.set_yticks(range(len(lbls)))
            ax.set_yticklabels(lbls, fontsize=10)
            ax.set_title(phase_name, fontsize=11)
            ax.grid(False)
 
            for i, (v, s) in enumerate(zip(vals.flatten(), stds)):
                ax.text(0, i, f"{v:.4f}\n+/-{s:.4f}",
                        ha="center", va="center", fontsize=9, color="black")
 
            best_i = int(np.argmax(vals.flatten()))
            ax.add_patch(mpatches.FancyBboxPatch(
                (-0.5, best_i - 0.5), 1, 1,
                boxstyle="round,pad=0.05",
                linewidth=2, edgecolor="#2166ac", facecolor="none"
            ))
 
        fig.tight_layout()
        save(fig, "04_heatmap_hyperparams")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 8 - Few-shot learning
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_fewshot():
    """
    Left: Val F1 learning curves per n_shots (mean +/- std).
    Right: grouped bar chart comparing Val vs Test F1 per shot count.
    """
    df_h = load_history("fewshot")
    df_r = load_results("fewshot")
 
    shots = sorted(df_h["n_shots"].unique())
 
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle("Phase 8: Few-Shot Learning - Custom CNN (CINIC-10)", fontsize=13)
 
        ax = axes[0]
        for i, n in enumerate(shots):
            sub = df_h[df_h["n_shots"] == n]
            s   = (sub.groupby("epoch")
                      .agg(val_f1_mean=("val_f1", "mean"), val_f1_std=("val_f1", "std"))
                      .reset_index())
            c = PALETTE[i]
            ax.plot(s["epoch"], s["val_f1_mean"], color=c, lw=2, label=f"{n} shots")
            ax.fill_between(s["epoch"],
                            s["val_f1_mean"] - s["val_f1_std"],
                            s["val_f1_mean"] + s["val_f1_std"],
                            color=c, alpha=0.15)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Val Macro F1")
        ax.set_title("Learning curves - Val F1 (mean +/- std)")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.legend(title="n shots")
 
        ax = axes[1]
        x     = np.arange(len(df_r))
        width = 0.35
        xlbls = [f"{n} shots\n(n={t})" for n, t in
                 zip(df_r["n_shots"], df_r["total_samples"])]
 
        b1 = ax.bar(x - width / 2, df_r["val_f1_mean"],  width, yerr=df_r["val_f1_std"],
                    capsize=5, label="Val F1",  color="#2166ac", alpha=0.82,
                    error_kw=dict(lw=1.3, capthick=1.3))
        b2 = ax.bar(x + width / 2, df_r["test_f1_mean"], width, yerr=df_r["test_f1_std"],
                    capsize=5, label="Test F1", color="#d6604d", alpha=0.82,
                    error_kw=dict(lw=1.3, capthick=1.3))
 
        for bar, v, s in zip(list(b1) + list(b2),
                              list(df_r["val_f1_mean"])  + list(df_r["test_f1_mean"]),
                              list(df_r["val_f1_std"])   + list(df_r["test_f1_std"])):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + s + 0.003,
                    f"{v:.3f}", ha="center", fontsize=9)
 
        ax.set_xticks(x)
        ax.set_xticklabels(xlbls)
        ax.set_ylabel("Macro F1")
        ax.set_title("Val vs Test F1 - few-shot (mean +/- std)")
        ax.legend()
 
        fig.tight_layout()
        save(fig, "05_fewshot")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# CROSS-PHASE SUMMARY - optimization progress
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_progress_across_phases():
    """
    Bar chart showing the best Test F1 achieved at each experimental phase,
    with a trend line to visualize cumulative optimization gains.
    """
    r_lr = load_results("lr")
    r_bs = load_results("bs")
    r_do = load_results("dropout")
    r_wd = load_results("wd")
    r_a5 = load_results("aug_std")
    r_a6 = load_results("aug_adv")
    r_a7 = load_results("aug_mix")
 
    def best(df):
        row = df.loc[df["test_f1_mean"].idxmax()]
        return float(row["test_f1_mean"]), float(row["test_f1_std"])
 
    stages = [
        ("Baseline\n(LR=1e-2, BS=64)", *best(r_lr.head(1))),
        ("LR sweep",                    *best(r_lr)),
        ("BS sweep",                    *best(r_bs)),
        ("+ Dropout",                   *best(r_do)),
        ("+ Weight Decay",              *best(r_wd)),
        ("+ Aug std\n(flip)",           *best(r_a5)),
        ("+ CutMix",                    *best(r_a6)),
        ("+ CutOut/Random",             *best(r_a7)),
    ]
    labels = [s[0] for s in stages]
    means  = [s[1] for s in stages]
    stds   = [s[2] for s in stages]
 
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(12, 5))
 
        colors = ["#b0bec5" if m < max(means) else "#2166ac" for m in means]
        bars = ax.bar(range(len(stages)), means, yerr=stds, capsize=5,
                      color=colors, alpha=0.85,
                      error_kw=dict(lw=1.4, capthick=1.4))
 
        ax.plot(range(len(stages)), means, "o--", color="#d6604d",
                lw=1.5, ms=5, zorder=5)
 
        for bar, m, s in zip(bars, means, stds):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + s + 0.003,
                    f"{m:.4f}", ha="center", fontsize=9)
 
        ax.set_xticks(range(len(stages)))
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel("Test Macro F1")
        ax.set_title("Optimization progress - best Test F1 per phase (Custom CNN)")
        ax.set_ylim(min(means) - 0.03, max(means) + 0.04)
 
        delta = max(means) - means[0]
        ax.annotate(f"+{delta:.4f} vs baseline",
                    xy=(np.argmax(means), max(means) + max(stds) + 0.008),
                    fontsize=10, color="#2166ac", ha="center")
 
        fig.tight_layout()
        save(fig, "06_progress_across_phases")

In [8]:
print(f"Saving plots to: {OUTPUT_DIR.resolve()}\n")

print("1/6  LR + BS sweep...")
plot_hyperparams_sweep()

print("2/6  Regularization (Dropout + WD)...")
plot_regularization()

print("3/6  Augmentations...")
plot_augmentations()

print("4/6  Hyperparameter heatmap...")
plot_heatmap_hyperparam()

print("5/6  Few-shot learning...")
plot_fewshot()

print("6/6  Progress across phases...")
plot_progress_across_phases()

print("\nDone!")

Saving plots to: C:\Semestr_8\DeepLearning\plots

1/6  LR + BS sweep...
  v plots\01_lr_bs_sweep.png
2/6  Regularization (Dropout + WD)...
  v plots\02_regularization.png
3/6  Augmentations...
  v plots\03_augmentations.png
4/6  Hyperparameter heatmap...


C:\Users\Mikolaj\AppData\Local\Temp\ipykernel_2656\3612043146.py:360: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  v plots\04_heatmap_hyperparams.png
5/6  Few-shot learning...
  v plots\05_fewshot.png
6/6  Progress across phases...
  v plots\06_progress_across_phases.png

Done!
